In [3]:
import mysql.connector

conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="admin",
    database="rsanalysis"
)

cursor = conn.cursor()

print("Connected successfully!")

Connected successfully!


In [27]:
from faker import Faker
import random

fake = Faker()

customers = []

for i in range(1, 101):  # 100 customers
    customers.append((
        fake.first_name(),
        fake.last_name(),
        random.choice(['Male', 'Female', 'Other']),
        fake.date_of_birth(minimum_age=18, maximum_age=60),
        fake.email(),
        str(random.randint(7000000000, 9999999999)),
        fake.city(),
        fake.state(),
        fake.date_this_year(),
        random.choice(['Regular' ,'Silver', 'Gold', 'Platinum']),
        random.choices(
            population = [True, False],
            weights = [85,15],
            k =1
        )[0]
        
    ))

In [29]:
sql = """
INSERT INTO customers (
first_name, last_name, gender, date_of_birth,
email, phone_number, city, state, join_date,membership_type,is_active
)
VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
"""

cursor.executemany(sql, customers)
conn.commit()

In [33]:
categories = [
    ("Electronics",),
    ("Clothing",),
    ("Home & Kitchen",),
    ("Beauty",),
    ("Grocery",),
    ("Sports",),
    ("Books",)
]

In [35]:
sql = """
INSERT INTO category (category_name)
VALUES (%s)
"""

cursor.executemany(sql, categories)
conn.commit()

In [87]:
from faker import Faker
import random

fake = Faker()
suppliers = []

for i in range(20):
    suppliers.append((
        fake.company(),
        fake.name(),
        str(random.randint(7000000000, 9999999999)),
        f"supplier{i}@example.com",
        fake.city(),
        fake.state(),
        True
    ))

sql_suppliers = """
INSERT INTO suppliers (
    supplier_name,
    contact_person,
    contact_number,
    contact_email,
    city,
    state,
    is_active
)
VALUES (%s, %s, %s, %s, %s, %s, %s)
"""

cursor.executemany(sql_suppliers, suppliers)
conn.commit()

print("Suppliers inserted successfully")

Suppliers inserted successfully


In [107]:
from faker import Faker
import random

fake = Faker()

# 1. Fetch categories
cursor.execute("SELECT category_id, category_name FROM category")
categories = cursor.fetchall()

# 2. Fetch suppliers
cursor.execute("SELECT supplier_id FROM suppliers")
supplier_ids = [row[0] for row in cursor.fetchall()]

print("Categories:", categories)
print("Supplier IDs:", supplier_ids)

product_templates = {
    "electronics": ["Phone", "Laptop", "Headphones", "Tablet", "Smartwatch"],
    "clothing": ["T-Shirt", "Jeans", "Jacket", "Shirt", "Hoodie"],
    "home & kitchen": ["Mixer", "Knife Set", "Cookware", "Blender"],
    "beauty": ["Cream", "Perfume", "Lotion", "Shampoo"],
    "grocery": ["Rice", "Oil", "Pasta", "Snacks"],
    "sports": ["Football", "Bat", "Racket", "Shoes"],
    "books": ["Novel", "Notebook", "Biography", "Comics"]
}

products = []

for category_id, category_name in categories:
    normalized_category = category_name.strip().lower()

    if normalized_category not in product_templates:
        print("Skipped category:", category_name)
        continue

    for i, base_name in enumerate(product_templates[normalized_category], start=1):
        product_name = f"{base_name} {fake.word().title()}"
        brand = fake.company()
        supplier_id = random.choice(supplier_ids)

        price = round(random.uniform(100, 5000), 2)
        cost_price = round(price * random.uniform(0.55, 0.8), 2)
        stock_quantity = random.randint(20, 300)
        sku = f"{normalized_category[:3].upper()}-{i:03d}-{random.randint(100,999)}"

        products.append((
            product_name,
            category_id,
            price,
            brand,
            sku,
            stock_quantity,
            cost_price,
            supplier_id
        ))

print("Generated products:", len(products))
print("Sample:", products[:3])

sql_products = """
INSERT INTO product (
    product_name,
    category_id,
    price,
    brand,
    sku,
    stock_quantity,
    cost_price,
    supplier_id
)
VALUES (%s, %s, %s, %s, %s, %s, %s, %s)
"""

cursor.executemany(sql_products, products)
conn.commit()

print("Products inserted successfully!")

Categories: [(4, 'Beauty'), (7, 'Books'), (2, 'Clothing'), (1, 'Electronics'), (5, 'Grocery'), (3, 'Home & Kitchen'), (6, 'Sports')]
Supplier IDs: [19, 2, 13, 3, 20, 11, 16, 17, 7, 10, 9, 4, 15, 8, 18, 12, 14, 1, 6, 5]
Generated products: 30
Sample: [('Cream Red', 4, 123.54, 'Brooks, Morris and King', 'BEA-001-272', 32, 86.21, 8), ('Perfume Difference', 4, 1573.77, 'Thomas, Williams and Collins', 'BEA-002-565', 47, 984.91, 10), ('Lotion Skill', 4, 843.55, 'Banks, Stout and Johnson', 'BEA-003-784', 200, 585.16, 7)]
Products inserted successfully!


In [109]:
from faker import Faker
import random

fake = Faker()

# Fetch customer IDs from DB
cursor.execute("SELECT cust_id FROM customers")
customer_ids = [row[0] for row in cursor.fetchall()]

print("Customers found:", len(customer_ids))

orders = []

order_statuses = ['Delivered', 'Shipped', 'Pending', 'Cancelled']
status_weights = [70, 15, 10, 5]

num_orders = 500

for _ in range(num_orders):
    customer_id = random.choice(customer_ids)
    order_date = fake.date_time_between(start_date='-12M', end_date='now')
    status = random.choices(order_statuses, weights=status_weights, k=1)[0]

    orders.append((customer_id, order_date, status))

print("Generated orders:", len(orders))
print("Sample orders:", orders[:3])

sql_orders = """
INSERT INTO orders (
    customer_id,
    order_date,
    status
)
VALUES (%s, %s, %s)
"""

cursor.executemany(sql_orders, orders)
conn.commit()

print("Orders inserted successfully!")

Customers found: 100
Generated orders: 500
Sample orders: [(180, datetime.datetime(2025, 8, 6, 22, 5, 27), 'Delivered'), (185, datetime.datetime(2026, 2, 26, 10, 12, 9), 'Pending'), (193, datetime.datetime(2026, 5, 28, 10, 5, 7), 'Delivered')]
Orders inserted successfully!


In [111]:
# Fetch order IDs
cursor.execute("SELECT order_id FROM orders")
order_ids = [row[0] for row in cursor.fetchall()]

# Fetch product IDs + price
cursor.execute("SELECT product_id, price FROM product")
product_data = cursor.fetchall()

product_price_map = {product_id: price for product_id, price in product_data}
product_ids = list(product_price_map.keys())

print("Orders found:", len(order_ids))
print("Products found:", len(product_ids))

order_items = []

for order_id in order_ids:
    num_items = random.randint(1, 5)

    # pick unique products for that order
    chosen_products = random.sample(product_ids, min(num_items, len(product_ids)))

    for product_id in chosen_products:
        quantity = random.randint(1, 3)
        price_at_purchase = product_price_map[product_id]

        order_items.append((
            order_id,
            product_id,
            quantity,
            price_at_purchase
        ))

print("Generated order items:", len(order_items))
print("Sample order items:", order_items[:5])

sql_order_items = """
INSERT INTO order_items (
    order_id,
    product_id,
    quantity,
    price_at_purchase
)
VALUES (%s, %s, %s, %s)
"""

cursor.executemany(sql_order_items, order_items)
conn.commit()

print("Order items inserted successfully!")

Orders found: 500
Products found: 30
Generated order items: 1498
Sample order items: [(276, 1, 2, Decimal('123.54')), (276, 22, 2, Decimal('4023.19')), (276, 24, 3, Decimal('1421.34')), (276, 29, 3, Decimal('1639.43')), (276, 14, 1, Decimal('4998.82'))]
Order items inserted successfully!


In [113]:
cursor.execute("""
SELECT 
    order_id,
    SUM(quantity * price_at_purchase) AS total_amount
FROM order_items
GROUP BY order_id
""")

order_totals = cursor.fetchall()

print("Orders with totals:", len(order_totals))
print("Sample:", order_totals[:5])

Orders with totals: 500
Sample: [(1, Decimal('18173.71')), (2, Decimal('14248.77')), (3, Decimal('17086.92')), (4, Decimal('7247.86')), (5, Decimal('10900.77'))]


In [115]:
from faker import Faker
import random

fake = Faker()

payments = []

payment_modes = ['Cash', 'UPI', 'Net Banking', 'Credit Card', 'Debit Card']
payment_mode_weights = [10, 40, 10, 20, 20]

payment_statuses = ['Successful', 'Failed', 'Refunded', 'Pending']
payment_status_weights = [80, 8, 7, 5]

for order_id, total_amount in order_totals:
    payment_mode = random.choices(payment_modes, weights=payment_mode_weights, k=1)[0]
    payment_status = random.choices(payment_statuses, weights=payment_status_weights, k=1)[0]

    # Cash can have no transaction_id, others usually have one
    if payment_mode == 'Cash':
        transaction_id = None
    else:
        transaction_id = fake.uuid4()

    payment_date = fake.date_time_between(start_date='-12M', end_date='now')

    payments.append((
        order_id,
        payment_mode,
        float(total_amount),
        transaction_id,
        payment_status,
        payment_date
    ))

print("Generated payments:", len(payments))
print("Sample payments:", payments[:5])

Generated payments: 500
Sample payments: [(1, 'Cash', 18173.71, None, 'Successful', datetime.datetime(2026, 2, 15, 13, 58, 31)), (2, 'UPI', 14248.77, 'caaeacdd-1d7a-4fa9-9e63-0fa82e5097ae', 'Successful', datetime.datetime(2026, 3, 15, 3, 19, 26)), (3, 'Debit Card', 17086.92, '6c727270-772d-4d2c-8819-39fd5fbf1c30', 'Successful', datetime.datetime(2026, 2, 27, 20, 45, 55)), (4, 'Cash', 7247.86, None, 'Successful', datetime.datetime(2025, 9, 17, 20, 14, 27)), (5, 'Net Banking', 10900.77, '6e0b1e63-b14a-4d64-945a-0673a3f944e0', 'Successful', datetime.datetime(2026, 3, 12, 8, 46, 40))]


In [117]:
sql_payments = """
INSERT INTO payments (
    order_id,
    payment_mode,
    amount,
    transaction_id,
    payment_status,
    payment_date
)
VALUES (%s, %s, %s, %s, %s, %s)
"""

cursor.executemany(sql_payments, payments)
conn.commit()

print("Payments inserted successfully!")

Payments inserted successfully!


In [119]:
cursor.execute("""
SELECT 
    order_item_id,
    quantity,
    price_at_purchase
FROM order_items
""")

order_items_data = cursor.fetchall()

print("Order items found:", len(order_items_data))
print("Sample:", order_items_data[:5])

Order items found: 1498
Sample: [(1, 2, Decimal('123.54')), (2, 2, Decimal('4023.19')), (3, 3, Decimal('1421.34')), (4, 3, Decimal('1639.43')), (5, 1, Decimal('4998.82'))]


In [121]:
from faker import Faker
import random

fake = Faker()

returns = []

return_reasons = [
    "Damaged product",
    "Wrong item delivered",
    "Defective item",
    "Quality not as expected",
    "Changed mind",
    "Size issue",
    "Late delivery"
]

return_statuses = ['Pending', 'Approved', 'Rejected', 'Completed']
return_status_weights = [10, 20, 10, 60]

for order_item_id, quantity, price_at_purchase in order_items_data:
    # only some order items get returned
    if random.random() < 0.10:   # 10% return probability
        returned_quantity = random.randint(1, quantity)
        return_amount = returned_quantity * float(price_at_purchase)
        return_reason = random.choice(return_reasons)
        return_status = random.choices(return_statuses, weights=return_status_weights, k=1)[0]
        return_date = fake.date_time_between(start_date='-6M', end_date='now')

        returns.append((
            order_item_id,
            returned_quantity,
            return_amount,
            return_reason,
            return_status,
            return_date
        ))

print("Generated returns:", len(returns))
print("Sample returns:", returns[:5])

Generated returns: 149
Sample returns: [(1, 2, 247.08, 'Quality not as expected', 'Approved', datetime.datetime(2026, 4, 25, 2, 24, 17)), (5, 1, 4998.82, 'Defective item', 'Completed', datetime.datetime(2026, 4, 8, 9, 55, 20)), (6, 1, 907.23, 'Quality not as expected', 'Pending', datetime.datetime(2026, 3, 7, 11, 57, 33)), (9, 1, 1419.27, 'Late delivery', 'Approved', datetime.datetime(2026, 1, 17, 22, 39, 8)), (36, 1, 147.62, 'Wrong item delivered', 'Approved', datetime.datetime(2026, 1, 3, 9, 25, 55))]


In [123]:
sql_returns = """
INSERT INTO returns (
    order_item_id,
    returned_quantity,
    return_amount,
    return_reason,
    return_status,
    return_date
)
VALUES (%s, %s, %s, %s, %s, %s)
"""

cursor.executemany(sql_returns, returns)
conn.commit()

print("Returns inserted successfully!")

Returns inserted successfully!


In [19]:
cursor.execute("SELECT * FROM store;")
print(cursor.fetchall())

[(1, 'RetailMart Manhattan', 'New York', 'New York', '2125551001', 'manhattan@retailmart.com', 1, datetime.datetime(2026, 7, 3, 11, 10, 41), datetime.datetime(2026, 7, 3, 11, 10, 41)), (2, 'RetailMart Brooklyn', 'Brooklyn', 'New York', '2125551002', 'brooklyn@retailmart.com', 1, datetime.datetime(2026, 7, 3, 11, 10, 41), datetime.datetime(2026, 7, 3, 11, 10, 41)), (3, 'RetailMart Los Angeles Central', 'Los Angeles', 'California', '3105551003', 'la.central@retailmart.com', 1, datetime.datetime(2026, 7, 3, 11, 10, 41), datetime.datetime(2026, 7, 3, 11, 10, 41)), (4, 'RetailMart San Diego Plaza', 'San Diego', 'California', '6195551004', 'sandiego@retailmart.com', 1, datetime.datetime(2026, 7, 3, 11, 10, 41), datetime.datetime(2026, 7, 3, 11, 10, 41)), (5, 'RetailMart Houston Hub', 'Houston', 'Texas', '7135551005', 'houston@retailmart.com', 1, datetime.datetime(2026, 7, 3, 11, 10, 41), datetime.datetime(2026, 7, 3, 11, 10, 41)), (6, 'RetailMart Dallas Square', 'Dallas', 'Texas', '214555100

In [29]:
stores = [
    ("RetailMart Manhattan", "New York", "New York", "2125551001", "manhattan@retailmart.com", "2025-05-16", True),
    ("RetailMart Brooklyn", "Brooklyn", "New York", "2125551002", "brooklyn@retailmart.com", "2021-03-05", True),
    ("RetailMart Los Angeles Central", "Los Angeles", "California", "3105551003", "la.central@retailmart.com", "2025-12-06", True),
    ("RetailMart San Diego Plaza", "San Diego", "California", "6195551004", "sandiego@retailmart.com", "2026-01-17", True),
    ("RetailMart Houston Hub", "Houston", "Texas", "7135551005", "houston@retailmart.com", "2023-09-01", True),
    ("RetailMart Dallas Square", "Dallas", "Texas", "2145551006", "dallas@retailmart.com", "2025-05-16", True),
    ("RetailMart Chicago Downtown", "Chicago", "Illinois", "3125551007", "chicago@retailmart.com", "2024-03-12", True),
    ("RetailMart Miami Bay", "Miami", "Florida", "3055551008", "miami@retailmart.com", "2025-07-26", True)
]

sql_stores = """
INSERT INTO stores (
    store_name,
    city,
    state,
    store_contact,
    store_email,
    opening_date,
    is_active
)
VALUES (%s, %s, %s, %s, %s, %s, %s)
"""

cursor.executemany(sql_stores, stores)
conn.commit()

print("Stores inserted successfully!")

Stores inserted successfully!


In [31]:
from faker import Faker
import random

fake = Faker()

# Fetch store IDs
cursor.execute("SELECT store_id FROM store")
store_ids = [row[0] for row in cursor.fetchall()]

print("Stores found:", len(store_ids))

employees = []

designations = [
    "Store Manager",
    "Cashier",
    "Sales Associate",
    "Inventory Executive",
    "Customer Support"
]

designation_weights = [10, 25, 35, 15, 15]

for i in range(50):   # create 50 employees
    first_name = fake.first_name()
    last_name = fake.last_name()
    email = f"{first_name.lower()}.{last_name.lower()}{i}@retailmart.com"
    contact_number = str(random.randint(7000000000, 9999999999))
    designation = random.choices(designations, weights=designation_weights, k=1)[0]

    if designation == "Store Manager":
        salary = round(random.uniform(45000, 70000), 2)
    elif designation == "Inventory Executive":
        salary = round(random.uniform(25000, 40000), 2)
    elif designation == "Customer Support":
        salary = round(random.uniform(22000, 35000), 2)
    elif designation == "Cashier":
        salary = round(random.uniform(18000, 28000), 2)
    else:  # Sales Associate
        salary = round(random.uniform(18000, 30000), 2)

    date_joined = fake.date_between(start_date='-5y', end_date='today')
    city = fake.city()
    state = fake.state()
    is_active = random.choices([True, False], weights=[90, 10], k=1)[0]
    store_id = random.choice(store_ids)

    employees.append((
        first_name,
        last_name,
        email,
        contact_number,
        designation,
        salary,
        date_joined,
        city,
        state,
        is_active,
        store_id
    ))

print("Generated employees:", len(employees))
print("Sample:", employees[:3])

sql_employees = """
INSERT INTO employees (
    first_name,
    last_name,
    email,
    contact_number,
    designation,
    salary,
    date_joined,
    city,
    state,
    is_active,
    store_id
)
VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
"""

cursor.executemany(sql_employees, employees)
conn.commit()

print("Employees inserted successfully!")

Stores found: 8
Generated employees: 50
Sample: [('Sarah', 'Campbell', 'sarah.campbell0@retailmart.com', '8547794834', 'Sales Associate', 28539.55, datetime.date(2023, 4, 19), 'Ryanfort', 'California', True, 6), ('Jennifer', 'Harrison', 'jennifer.harrison1@retailmart.com', '9409666720', 'Customer Support', 28266.15, datetime.date(2022, 1, 5), 'West Brandonshire', 'New Jersey', True, 5), ('Theodore', 'Garcia', 'theodore.garcia2@retailmart.com', '9549983562', 'Inventory Executive', 36956.27, datetime.date(2025, 4, 4), 'East James', 'North Carolina', True, 4)]
Employees inserted successfully!
